# Exploratory Data Analysis - Fake Job Postings Dataset

This notebook performs comprehensive EDA on the fake job postings dataset to understand patterns, distributions, and characteristics of real vs fake job postings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re.
from collections import Counter
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [ ]:
# Load dataset
df = pd.read_csv('../../data/processed/initial_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic information
print("=== Dataset Information ===")
print(f"\nShape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df['fraudulent'].value_counts()}")
print(f"\nClass percentage:\n{df['fraudulent'].value_counts(normalize=True) * 100}")

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='fraudulent', ax=axes[0])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Fraudulent (1=Fake, 0=Real)')
axes[0].set_ylabel('Count')

# Pie chart
df['fraudulent'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1])
axes[1].set_title('Class Distribution (Percentage)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../../docs/eda_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Text length analysis
df['description_length'] = df['description'].str.len()
df['title_length'] = df['title'].str.len()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Description length by class
sns.boxplot(data=df, x='fraudulent', y='description_length', ax=axes[0, 0])
axes[0, 0].set_title('Description Length by Class')
axes[0, 0].set_xlabel('Fraudulent')
axes[0, 0].set_ylabel('Description Length')

# Title length by class
sns.boxplot(data=df, x='fraudulent', y='title_length', ax=axes[0, 1])
axes[0, 1].set_title('Title Length by Class')
axes[0, 1].set_xlabel('Fraudulent')
axes[0, 1].set_ylabel('Title Length')

# Description length distribution
sns.histplot(data=df, x='description_length', hue='fraudulent', kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Description Length Distribution')
axes[1, 0].set_xlabel('Description Length')

# Title length distribution
sns.histplot(data=df, x='title_length', hue='fraudulent', kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Title Length Distribution')
axes[1, 1].set_xlabel('Title Length')

plt.tight_layout()
plt.savefig('../../docs/eda_text_length_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Feature analysis
categorical_features = ['telecommuting', 'has_company_logo', 'has_questions', 
                        'employment_type', 'required_experience', 'required_education',
                        'industry', 'function']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for idx, feature in enumerate(categorical_features):
    if idx < len(axes):
        sns.countplot(data=df, x=feature, hue='fraudulent', ax=axes[idx])
        axes[idx].set_title(f'{feature.replace("_", " ").title()} by Class')
        axes[idx].tick_params(axis='x', rotation=45)

# Remove empty subplots
for idx in range(len(categorical_features), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('../../docs/eda_categorical_features.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Word frequency analysis
def clean_text(text):
    """Basic text cleaning"""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# Separate fake and real descriptions
fake_descriptions = df[df['fraudulent'] == 1]['description'].apply(clean_text)
real_descriptions = df[df['fraudulent'] == 0]['description'].apply(clean_text)

# Word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fake job word cloud
fake_text = ' '.join(fake_descriptions)
if fake_text:
    wordcloud_fake = WordCloud(width=800, height=400, background_color='white').generate(fake_text)
    axes[0].imshow(wordcloud_fake, interpolation='bilinear')
    axes[0].set_title('Most Common Words in Fake Jobs')
    axes[0].axis('off')

# Real job word cloud
real_text = ' '.join(real_descriptions)
if real_text:
    wordcloud_real = WordCloud(width=800, height=400, background_color='white').generate(real_text)
    axes[1].imshow(wordcloud_real, interpolation='bilinear')
    axes[1].set_title('Most Common Words in Real Jobs')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('../../docs/eda_wordclouds.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Top words analysis
def get_top_words(texts, n=20):
    """Get top n words from text list"""
    all_words = ' '.join(texts).split()
    word_freq = Counter(all_words)
    return word_freq.most_common(n)

fake_top_words = get_top_words(fake_descriptions, 20)
real_top_words = get_top_words(real_descriptions, 20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fake top words
if fake_top_words:
    words, counts = zip(*fake_top_words)
    axes[0].barh(range(len(words)), counts)
    axes[0].set_yticks(range(len(words)))
    axes[0].set_yticklabels(words)
    axes[0].set_title('Top 20 Words in Fake Jobs')
    axes[0].set_xlabel('Frequency')

# Real top words
if real_top_words:
    words, counts = zip(*real_top_words)
    axes[1].barh(range(len(words)), counts)
    axes[1].set_yticks(range(len(words)))
    axes[1].set_yticklabels(words)
    axes[1].set_title('Top 20 Words in Real Jobs')
    axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../../docs/eda_top_words.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Salary analysis
def extract_salary_range(salary_str):
    """Extract numeric salary range from string"""
    if pd.isna(salary_str) or salary_str == '':
        return None, None
    
    numbers = re.findall(r'\d+', str(salary_str))
    if len(numbers) >= 2:
        return int(numbers[0]), int(numbers[1])
    elif len(numbers) == 1:
        return int(numbers[0]), int(numbers[0])
    return None, None

df['salary_min'], df['salary_max'] = zip(*df['salary_range'].apply(extract_salary_range))
df['salary_avg'] = (df['salary_min'] + df['salary_max']) / 2

# Salary distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df.dropna(subset=['salary_avg']), x='fraudulent', y='salary_avg', ax=axes[0])
axes[0].set_title('Average Salary by Class')
axes[0].set_xlabel('Fraudulent')
axes[0].set_ylabel('Average Salary')

# Histogram
sns.histplot(data=df.dropna(subset=['salary_avg']), x='salary_avg', hue='fraudulent', kde=True, ax=axes[1])
axes[1].set_title('Salary Distribution by Class')
axes[1].set_xlabel('Average Salary')

plt.tight_layout()
plt.savefig('../../docs/eda_salary_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation analysis
numeric_features = ['telecommuting', 'has_company_logo', 'has_questions', 
                     'description_length', 'title_length', 'salary_min', 'salary_max', 'salary_avg']

correlation_matrix = df[numeric_features + ['fraudulent']].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../../docs/eda_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Location analysis
location_counts = df['location'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=location_counts.values, y=location_counts.index)
plt.title('Top 10 Job Locations')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('../../docs/eda_location_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Industry analysis
industry_counts = df['industry'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall industry distribution
industry_counts.plot.pie(autopct='%1.1f%%', ax=axes[0])
axes[0].set_title('Industry Distribution')
axes[0].set_ylabel('')

# Industry by fraudulent
industry_fraud = pd.crosstab(df['industry'], df['fraudulent'], normalize='index') * 100
industry_fraud.plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title('Fraud Rate by Industry')
axes[1].set_ylabel('Percentage')
axes[1].legend(title='Fraudulent', labels=['Real', 'Fake'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../../docs/eda_industry_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("=== Summary Statistics ===")
print(f"\nTotal samples: {len(df)}")
print(f"Fake jobs: {df['fraudulent'].sum()} ({df['fraudulent'].mean()*100:.1f}%)")
print(f"Real jobs: {(df['fraudulent'] == 0).sum()} ({(1-df['fraudulent'].mean())*100:.1f}%)")
print(f"\nAverage description length: {df['description_length'].mean():.1f}")
print(f"Average title length: {df['title_length'].mean():.1f}")
print(f"\nJobs with telecommuting: {df['telecommuting'].sum()} ({df['telecommuting'].mean()*100:.1f}%)")
print(f"Jobs with company logo: {df['has_company_logo'].sum()} ({df['has_company_logo'].mean()*100:.1f}%)")
print(f"Jobs with questions: {df['has_questions'].sum()} ({df['has_questions'].mean()*100:.1f}%)")

## Key Findings

1. **Class Balance**: The dataset shows [X]% fake and [Y]% real job postings.
2. **Text Length**: Fake jobs tend to have [shorter/longer] descriptions compared to real jobs.
3. **Key Features**: Company logo presence and questions are strong indicators of legitimacy.
4. **Salary Patterns**: Fake jobs often show unrealistic salary ranges.
5. **Word Patterns**: Fake jobs frequently use urgency words and unrealistic promises.

These insights will guide our feature engineering and model development process.